In [43]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

# Find the uploaded checkpoint regardless of its renamed filename
checkpoint_files = [
    name for name in uploaded
    if name.startswith("bilstm_crf") and name.endswith(".pt")
]

if not checkpoint_files:
    raise FileNotFoundError("Please upload the bilstm_crf checkpoint.")

source = checkpoint_files[0]

destination = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

os.makedirs(os.path.dirname(destination), exist_ok=True)

shutil.copy(source, destination)

print("Checkpoint copied successfully!")
print("Saved at:", destination)
print("File size (MB):", round(os.path.getsize(destination) / (1024 * 1024), 2))

Saving bilstm_crf.pt to bilstm_crf (3).pt
Checkpoint copied successfully!
Saved at: /content/drive/MyDrive/nlp_assignment/bilstm_crf.pt
File size (MB): 7.95


In [44]:
import os

CKPT2 = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

print("Checkpoint found:", os.path.exists(CKPT2))
print("Checkpoint size (MB):",
      round(os.path.getsize(CKPT2) / (1024 * 1024), 2))

Checkpoint found: True
Checkpoint size (MB): 7.95


In [45]:
# Cell 1 – Install and import dependencies
!pip -q install pytorch-crf

import os
import random
import urllib.request
import zipfile
import json
import math
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)


Device: cpu
PyTorch: 2.14.1+cpu


In [46]:
# Cell 2 – Download and load UD English EWT (CoNLL-U)
BASE = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"
os.makedirs("data", exist_ok=True)

for split in ["train", "dev", "test"]:
    filename = f"en_ewt-ud-{split}.conllu"
    filepath = os.path.join("data", filename)
    if not os.path.exists(filepath):
        print("Downloading", filename)
        urllib.request.urlretrieve(BASE + filename, filepath)
    else:
        print("Already downloaded:", filename)

def read_conllu(path):
    sentences, words, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if line.startswith("#"):
                continue
            if not line:
                if words:
                    sentences.append((words, tags))
                    words, tags = [], []
                continue

            cols = line.split("\t")
            # Ignore multiword-token rows (e.g. 1-2) and empty nodes (e.g. 3.1)
            if "-" in cols[0] or "." in cols[0]:
                continue
            words.append(cols[1])
            tags.append(cols[3])  # UPOS column

    if words:
        sentences.append((words, tags))
    return sentences

train = read_conllu("data/en_ewt-ud-train.conllu")
dev   = read_conllu("data/en_ewt-ud-dev.conllu")
test  = read_conllu("data/en_ewt-ud-test.conllu")

tag_names = sorted({tag for _, tags in train for tag in tags})
t2i = {tag: i for i, tag in enumerate(tag_names)}
i2t = {i: tag for tag, i in t2i.items()}
K = len(tag_names)

print("Train sentences:", len(train))
print("Dev sentences:", len(dev))
print("Test sentences:", len(test))
print("Tags:", K, tag_names)

assert K == 17, f"Expected 17 UPOS tags, found {K}"
assert all(len(ws) == len(ts) for split in (train, dev, test) for ws, ts in split)
print("Dataset loaded and validated.")


Already downloaded: en_ewt-ud-train.conllu
Already downloaded: en_ewt-ud-dev.conllu
Already downloaded: en_ewt-ud-test.conllu
Train sentences: 12544
Dev sentences: 2001
Test sentences: 2077
Tags: 17 ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
Dataset loaded and validated.


In [48]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

bundle_files = [
    name for name in uploaded
    if name.startswith("task2_vocab_bundle") and name.endswith(".pt")
]

if not bundle_files:
    raise FileNotFoundError("Please upload task2_vocab_bundle.pt")

os.makedirs("/content/drive/MyDrive/nlp_assignment", exist_ok=True)

destination = "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"

shutil.copy(bundle_files[0], destination)

print("Vocabulary bundle copied successfully!")
print("File size (MB):", round(os.path.getsize(destination) / (1024 * 1024), 2))

Saving task2_vocab_bundle.pt to task2_vocab_bundle.pt
Vocabulary bundle copied successfully!
File size (MB): 7.48


In [49]:
import os

print("Bundle exists:", os.path.exists(
    "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"
))

Bundle exists: True


In [50]:
import torch

bundle_path = "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"

task2_bundle = torch.load(
    bundle_path,
    map_location="cpu",
    weights_only=False
)

w2i = task2_bundle["w2i"]
c2i = task2_bundle["c2i"]
emb = task2_bundle["emb"]

print("Word vocabulary:", len(w2i))
print("Character vocabulary:", len(c2i))
print("Embedding shape:", emb.shape)

Word vocabulary: 18643
Character vocabulary: 110
Embedding shape: torch.Size([18643, 100])


In [54]:
# Cell 3 – Load exact Task 2 vocabulary and embeddings

import torch
import os

bundle_path = "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"

if not os.path.exists(bundle_path):
    raise FileNotFoundError(
        "Task 2 vocabulary bundle not found. Check Google Drive."
    )

task2_bundle = torch.load(
    bundle_path,
    map_location="cpu",
    weights_only=False
)

w2i = task2_bundle["w2i"]
c2i = task2_bundle["c2i"]
emb = task2_bundle["emb"]

# Recreate vocabulary lists using the original indices
words = [None] * len(w2i)
for word, idx in w2i.items():
    words[idx] = word

chars = [None] * len(c2i)
for char, idx in c2i.items():
    chars[idx] = char

print("Task 2 vocabulary loaded successfully!")
print("Word vocabulary:", len(w2i))
print("Character vocabulary:", len(c2i))
print("Embedding matrix:", tuple(emb.shape))

assert len(w2i) == 18643
assert len(c2i) == 110
assert emb.shape == (18643, 100)

Task 2 vocabulary loaded successfully!
Word vocabulary: 18643
Character vocabulary: 110
Embedding matrix: (18643, 100)


In [55]:
# Cell 4 – BiLSTM encoder and Task 2 BiLSTM-CRF architecture
class BiLSTMEncoder(nn.Module):
    """Character BiLSTM + GloVe word embeddings + word-level BiLSTM."""
    def __init__(self, emb_matrix, n_chars,
                 char_dim=25, char_hid=25, word_hid=100, dropout=0.5):
        super().__init__()
        self.word_emb = nn.Embedding.from_pretrained(
            emb_matrix.clone(), freeze=False, padding_idx=0
        )
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.char_lstm = nn.LSTM(
            char_dim, char_hid, batch_first=True, bidirectional=True
        )
        self.word_lstm = nn.LSTM(
            emb_matrix.size(1) + 2 * char_hid,
            word_hid, batch_first=True, bidirectional=True
        )
        self.drop = nn.Dropout(dropout)
        self.out_dim = 2 * word_hid

    def forward(self, W, C, lens):
        B, T, L = C.shape
        char_lengths = (C != 0).sum(-1).clamp(min=1).reshape(-1)
        char_emb = self.drop(self.char_emb(C.reshape(B * T, L)))
        packed_chars = pack_padded_sequence(
            char_emb, char_lengths.cpu(), batch_first=True,
            enforce_sorted=False
        )
        _, (hidden, _) = self.char_lstm(packed_chars)
        char_vectors = torch.cat([hidden[0], hidden[1]], dim=-1).reshape(B, T, -1)

        x = self.drop(torch.cat([self.word_emb(W), char_vectors], dim=-1))
        packed_words = pack_padded_sequence(
            x, lens.cpu(), batch_first=True, enforce_sorted=False
        )
        output, _ = self.word_lstm(packed_words)
        output, _ = pad_packed_sequence(
            output, batch_first=True, total_length=T
        )
        return self.drop(output)


class BiLSTMCRF(nn.Module):
    """Task 2 model, needed to load and evaluate the saved Task 2 checkpoint."""
    def __init__(self, encoder, n_tags):
        super().__init__()
        self.enc = encoder
        self.fc = nn.Linear(encoder.out_dim, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def emissions(self, W, C, lens):
        return self.fc(self.enc(W, C, lens))

    def loss(self, W, C, Y, lens, mask):
        return -self.crf(self.emissions(W, C, lens), Y, mask=mask, reduction="mean")

    def decode(self, W, C, lens, mask):
        return self.crf.decode(self.emissions(W, C, lens), mask=mask)


print("BiLSTMEncoder and BiLSTMCRF defined.")


BiLSTMEncoder and BiLSTMCRF defined.


In [56]:
# Cell 5 – Batching, prediction and accuracy helpers
MAXC = 20
BS = 16

def collate(batch, train_mode=False):
    B = len(batch)
    T = max(len(ws) for ws, _ in batch)
    L = max(1, max(min(len(w), MAXC) for ws, _ in batch for w in ws))

    W = torch.zeros(B, T, dtype=torch.long)
    C = torch.zeros(B, T, L, dtype=torch.long)
    Y = torch.zeros(B, T, dtype=torch.long)
    lens = torch.tensor([len(ws) for ws, _ in batch], dtype=torch.long)

    for b, (ws, ts) in enumerate(batch):
        for t, (w, tag) in enumerate(zip(ws, ts)):
            lw = w.lower()
            wi = w2i.get(lw, 1)

            # Task 2 singleton replacement used during training only.
            if train_mode and word_counts[lw] == 1 and random.random() < 0.5:
                wi = 1

            W[b, t] = wi
            Y[b, t] = t2i[tag]

            for j, ch in enumerate(w[:MAXC]):
                C[b, t, j] = c2i.get(ch, 1)

    mask = torch.arange(T)[None, :] < lens[:, None]
    return W.to(device), C.to(device), Y.to(device), lens, mask.to(device)


def predict_crf(model, data, batch_size=64):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(data), batch_size):
            batch = data[start:start + batch_size]
            W, C, Y, lens, mask = collate(batch)
            decoded = model.decode(W, C, lens, mask)
            predictions.extend([[tag_names[j] for j in seq] for seq in decoded])
    return predictions


def token_accuracy(predictions, data):
    correct = total = 0
    for pred, (_, gold) in zip(predictions, data):
        correct += sum(p == g for p, g in zip(pred, gold))
        total += len(gold)
    return correct / total if total else 0.0


print("Batching and evaluation helpers defined.")


Batching and evaluation helpers defined.


In [57]:
# Cell 6 – HMM tagger from Task 1
OPEN = {"ADJ", "ADV", "INTJ", "NOUN", "PROPN", "VERB", "X"}

def suffix_key(word, max_n=3):
    lower = word.lower()
    capitalized = word[0].isupper() if word else False
    return [(lower[-n:], capitalized)
            for n in range(min(max_n, len(lower)), 0, -1)]


class HMMTagger:
    def __init__(self, lam=0.1, rare_thresh=5):
        self.lam = lam
        self.rare = rare_thresh

    def fit(self, data):
        lam = self.lam
        self.tags = sorted({tag for _, tags in data for tag in tags})
        K_local = len(self.tags)
        tag_to_idx = {tag: i for i, tag in enumerate(self.tags)}
        self.K = K_local

        transitions = np.zeros((K_local + 1, K_local + 1))
        emissions = defaultdict(lambda: np.zeros(K_local))
        tag_counts = np.zeros(K_local)
        word_freq = Counter(w.lower() for ws, _ in data for w in ws)
        suffix_counts = defaultdict(lambda: np.zeros(K_local))

        for ws, ts in data:
            previous = K_local  # START
            for word, tag in zip(ws, ts):
                idx = tag_to_idx[tag]
                transitions[previous, idx] += 1
                emissions[word.lower()][idx] += 1
                tag_counts[idx] += 1

                if word_freq[word.lower()] <= self.rare:
                    for key in suffix_key(word):
                        suffix_counts[key][idx] += 1
                previous = idx

            transitions[previous, K_local] += 1  # END

        self.log_trans = np.log(
            (transitions + lam) /
            (transitions.sum(axis=1, keepdims=True) + lam * (K_local + 1))
        )

        vocab_size = len(emissions)
        self.emit = {
            word: np.log((counts + lam) /
                         (tag_counts + lam * (vocab_size + 1)))
            for word, counts in emissions.items()
        }
        self.suf = {
            key: np.log((counts + lam) /
                        (tag_counts + lam * len(suffix_counts)))
            for key, counts in suffix_counts.items()
        }

        self.open_mask = np.array([tag in OPEN for tag in self.tags])
        n_open = max(1, int(self.open_mask.sum()))
        self.unif = np.where(self.open_mask,
                             np.log(1.0 / n_open),
                             np.log(1e-8))
        return self

    def emission(self, word):
        lower = word.lower()
        if lower in self.emit:
            return self.emit[lower]
        for key in suffix_key(word):
            if key in self.suf:
                return self.suf[key]
        return self.unif

    def tag(self, words):
        if not words:
            return []
        emissions = np.asarray([self.emission(w) for w in words])
        path = viterbi(emissions, self.log_trans, self.K)
        return [self.tags[i] for i in path]


def viterbi(emission_scores, log_transitions, n_tags):
    """Viterbi with START/END states; emission_scores has shape (n_words, K)."""
    n = emission_scores.shape[0]
    if n == 0:
        return []

    delta = np.full((n, n_tags), -np.inf)
    back = np.zeros((n, n_tags), dtype=np.int64)
    delta[0] = log_transitions[n_tags, :n_tags] + emission_scores[0]

    for pos in range(1, n):
        scores = delta[pos - 1][:, None] + log_transitions[:n_tags, :n_tags]
        back[pos] = scores.argmax(axis=0)
        delta[pos] = scores.max(axis=0) + emission_scores[pos]

    best = int((delta[-1] + log_transitions[:n_tags, n_tags]).argmax())
    path = [best]
    for pos in range(n - 1, 0, -1):
        best = int(back[pos, best])
        path.append(best)
    return list(reversed(path))


def seq_acc(predictions, data):
    return token_accuracy(predictions, data)


hmm = HMMTagger(lam=0.1).fit(train)
assert hmm.tags == tag_names, "HMM tag order differs from neural tag order."
log_trans = hmm.log_trans
hmm_preds = [hmm.tag(ws) for ws, _ in test]

# Verify the standalone Viterbi gives the same predictions as HMMTagger.tag.
viterbi_check = [
    [tag_names[j] for j in viterbi(
        np.asarray([hmm.emission(w) for w in ws]), log_trans, K
    )]
    for ws, _ in test
]
print("HMM test accuracy:", round(seq_acc(hmm_preds, test), 4))
print("Viterbi agrees with HMMTagger.tag:",
      all(a == b for a, b in zip(hmm_preds, viterbi_check)))


HMM test accuracy: 0.9192
Viterbi agrees with HMMTagger.tag: True


In [58]:
# =====================================================
# CELL 7: FIND TASK 2 CHECKPOINT IN GOOGLE DRIVE
# =====================================================

import os
from google.colab import drive

# Mount Google Drive if necessary
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Exact location where Task 2 model was saved
CKPT2 = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

print("Checking Task 2 checkpoint...\n")

if os.path.isfile(CKPT2):

    HAS_TASK2_CKPT = True

    size_mb = os.path.getsize(CKPT2) / (1024 * 1024)

    print("SUCCESS! Task 2 checkpoint found.")
    print("Path:", CKPT2)
    print("Size:", round(size_mb, 2), "MB")

else:

    HAS_TASK2_CKPT = False

    print("Checkpoint not found at expected location.")
    print("\nSearching MyDrive for model files...\n")

    for root, dirs, files in os.walk("/content/drive/MyDrive"):

        for file in files:

            if file.endswith((".pt", ".pth")):

                print(os.path.join(root, file))

    print("\nPlease check the printed locations.")

Checking Task 2 checkpoint...

SUCCESS! Task 2 checkpoint found.
Path: /content/drive/MyDrive/nlp_assignment/bilstm_crf.pt
Size: 7.95 MB


In [59]:

# ============================================================
# CELL 8: TASK 3 - NEURAL EMISSION HMM (SOFTMAX)
# Initialize encoder using trained Task 2 BiLSTM-CRF checkpoint
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
import random

# ------------------------------------------------------------
# 1. Check Task 2 checkpoint
# ------------------------------------------------------------

CKPT2 = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

if not os.path.exists(CKPT2):
    raise FileNotFoundError(
        "Task 2 checkpoint not found: " + CKPT2
    )

print("Task 2 checkpoint found!")

# ------------------------------------------------------------
# 2. Define Task 3 Softmax model
# ------------------------------------------------------------

class BiLSTMSoftmax(nn.Module):

    def __init__(self, encoder, n_tags):
        super().__init__()

        self.enc = encoder
        self.fc = nn.Linear(encoder.out_dim, n_tags)

    def forward(self, W, C, lens):
        return self.fc(self.enc(W, C, lens))


# ------------------------------------------------------------
# 3. Extract neural logits
# ------------------------------------------------------------

def get_logits(model, data, batch_size=64):

    model.eval()
    all_logits = []

    with torch.no_grad():

        for start in range(0, len(data), batch_size):

            batch = data[start:start + batch_size]

            W, C, Y, lens, mask = collate(batch)

            logits = model(W, C, lens).cpu().numpy()

            for b, length in enumerate(lens.tolist()):
                all_logits.append(logits[b, :length])

    return all_logits


# ------------------------------------------------------------
# 4. Calculate token accuracy
# ------------------------------------------------------------

def argmax_accuracy(logits_list, data):

    predictions = [
        [tag_names[int(j)] for j in logits.argmax(axis=-1)]
        for logits in logits_list
    ]

    return token_accuracy(predictions, data)


# ------------------------------------------------------------
# 5. Initialize Task 3 model
# ------------------------------------------------------------

model3 = BiLSTMSoftmax(
    BiLSTMEncoder(emb, len(chars)),
    K
).to(device)


# ------------------------------------------------------------
# 6. Load Task 2 trained encoder
# ------------------------------------------------------------

task2_state = torch.load(
    CKPT2,
    map_location=device
)

encoder_state = {
    key[len("enc."):]: value
    for key, value in task2_state.items()
    if key.startswith("enc.")
}

if not encoder_state:
    raise RuntimeError(
        "No encoder parameters found in Task 2 checkpoint."
    )

try:

    model3.enc.load_state_dict(
        encoder_state,
        strict=True
    )

    print("Task 2 trained encoder loaded successfully!")

except RuntimeError as e:

    raise RuntimeError(
        "Task 2 and Task 3 encoder architectures or vocabularies "
        "do not match. Check the word vocabulary, character "
        "vocabulary and embedding matrix.\n\n"
        + str(e)
    )


# ------------------------------------------------------------
# 7. Training configuration
# ------------------------------------------------------------

optimizer = torch.optim.SGD(
    model3.parameters(),
    lr=0.015,
    momentum=0.9
)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

BS = 16
MAX_EPOCHS = 50
PATIENCE = 5
LR0 = 0.015

best_dev = -1.0
bad_epochs = 0


# ------------------------------------------------------------
# 8. Save Task 3 model in Google Drive
# ------------------------------------------------------------

TASK3_DIR = "/content/drive/MyDrive/nlp_assignment"

os.makedirs(TASK3_DIR, exist_ok=True)

TASK3_CKPT = os.path.join(
    TASK3_DIR,
    "neural_softmax.pt"
)


# ------------------------------------------------------------
# 9. Train Task 3 model
# ------------------------------------------------------------

for epoch in range(MAX_EPOCHS):

    for group in optimizer.param_groups:
        group["lr"] = LR0 / (1 + 0.05 * epoch)

    random.shuffle(train)

    model3.train()

    total_loss = 0.0
    batches_seen = 0

    for start in range(0, len(train), BS):

        batch = train[start:start + BS]

        W, C, Y, lens, mask = collate(
            batch,
            train_mode=True
        )

        optimizer.zero_grad()

        logits = model3(W, C, lens)

        targets = Y.masked_fill(~mask, -100)

        loss = criterion(
            logits.reshape(-1, K),
            targets.reshape(-1)
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model3.parameters(),
            5.0
        )

        optimizer.step()

        total_loss += loss.item()
        batches_seen += 1

    # Evaluate on development set
    dev_logits_epoch = get_logits(model3, dev)

    dev_acc = argmax_accuracy(
        dev_logits_epoch,
        dev
    )

    mean_loss = total_loss / max(1, batches_seen)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {mean_loss:.4f} | "
        f"Dev Accuracy: {dev_acc:.4f}"
    )

    # Save best model
    if dev_acc > best_dev:

        best_dev = dev_acc
        bad_epochs = 0

        torch.save(
            model3.state_dict(),
            TASK3_CKPT
        )

        print("Best Task 3 model saved!")

    else:

        bad_epochs += 1

        if bad_epochs >= PATIENCE:

            print("Early stopping.")
            break


# ------------------------------------------------------------
# 10. Load best Task 3 model
# ------------------------------------------------------------

if not os.path.exists(TASK3_CKPT):

    raise RuntimeError(
        "Task 3 checkpoint was not saved."
    )

model3.load_state_dict(
    torch.load(TASK3_CKPT, map_location=device)
)

model3.eval()

print("\nTask 3 training completed!")
print("Best Task 3 dev accuracy:", round(best_dev, 4))
print("Task 3 checkpoint:", TASK3_CKPT)
print("Task 3 model saved successfully!")


Task 2 checkpoint found!
Task 2 trained encoder loaded successfully!
Epoch 01 | Loss: 0.3983 | Dev Accuracy: 0.9431
Best Task 3 model saved!
Epoch 02 | Loss: 0.2082 | Dev Accuracy: 0.9468
Best Task 3 model saved!
Epoch 03 | Loss: 0.1927 | Dev Accuracy: 0.9480
Best Task 3 model saved!
Epoch 04 | Loss: 0.1857 | Dev Accuracy: 0.9493
Best Task 3 model saved!
Epoch 05 | Loss: 0.1797 | Dev Accuracy: 0.9493
Best Task 3 model saved!
Epoch 06 | Loss: 0.1745 | Dev Accuracy: 0.9500
Best Task 3 model saved!
Epoch 07 | Loss: 0.1721 | Dev Accuracy: 0.9501
Best Task 3 model saved!
Epoch 08 | Loss: 0.1715 | Dev Accuracy: 0.9504
Best Task 3 model saved!
Epoch 09 | Loss: 0.1696 | Dev Accuracy: 0.9513
Best Task 3 model saved!
Epoch 10 | Loss: 0.1651 | Dev Accuracy: 0.9511
Epoch 11 | Loss: 0.1649 | Dev Accuracy: 0.9510
Epoch 12 | Loss: 0.1656 | Dev Accuracy: 0.9513
Epoch 13 | Loss: 0.1637 | Dev Accuracy: 0.9512
Epoch 14 | Loss: 0.1631 | Dev Accuracy: 0.9518
Best Task 3 model saved!
Epoch 15 | Loss: 0.1615

In [60]:
# Cell 9 – Save Task 3 checkpoint to Google Drive
task3_drive_dir = os.path.join(MYDRIVE, "nlp_assignment")
os.makedirs(task3_drive_dir, exist_ok=True)
drive_task3_ckpt = os.path.join(task3_drive_dir, "neural_softmax.pt")
torch.save(model3.state_dict(), drive_task3_ckpt)
print("Saved Task 3 checkpoint to:", drive_task3_ckpt)


Saved Task 3 checkpoint to: /content/drive/MyDrive/nlp_assignment/neural_softmax.pt


In [61]:
# Cell 10 – Temperature scaling and tag prior
def log_softmax_np(x, temperature=1.0):
    z = x / temperature
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))


dev_logits = get_logits(model3, dev)
test_logits = get_logits(model3, test)
gold_dev = [np.asarray([t2i[tag] for tag in tags], dtype=np.int64)
            for _, tags in dev]

def mean_nll(logits_list, gold_sequences, temperature):
    total_nll = 0.0
    total_tokens = 0
    for logits, gold in zip(logits_list, gold_sequences):
        log_probs = log_softmax_np(logits, temperature)
        total_nll -= log_probs[np.arange(len(gold)), gold].sum()
        total_tokens += len(gold)
    return total_nll / max(1, total_tokens)

temperature_grid = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
for temp in temperature_grid:
    print(f"T={temp:>4}: dev NLL={mean_nll(dev_logits, gold_dev, temp):.5f}")

best_T = min(temperature_grid,
             key=lambda temp: mean_nll(dev_logits, gold_dev, temp))
print("Selected temperature:", best_T)

tag_counts = Counter(tag for _, tags in train for tag in tags)
tag_prior = np.asarray([tag_counts[tag] for tag in tag_names], dtype=np.float64)
tag_prior /= tag_prior.sum()
log_prior = np.log(tag_prior)

print("Tag prior computed.")


T= 0.5: dev NLL=0.23730
T=0.75: dev NLL=0.17488
T= 1.0: dev NLL=0.15140
T=1.25: dev NLL=0.14635
T= 1.5: dev NLL=0.15397
T= 2.0: dev NLL=0.20100
T= 2.5: dev NLL=0.28537
T= 3.0: dev NLL=0.39812
Selected temperature: 1.25
Tag prior computed.


In [62]:
# Cell 11 – Neural emissions + unchanged Task 1 transitions + Viterbi
def hybrid_predict(logits_list, temperature=1.0, use_prior=True):
    predictions = []
    for logits in logits_list:
        emission_scores = log_softmax_np(logits, temperature)
        if use_prior:
            # log P(tag | context) - log P(tag) is proportional to log P(context | tag)
            emission_scores = emission_scores - log_prior
        path = viterbi(emission_scores, log_trans, K)
        predictions.append([tag_names[j] for j in path])
    return predictions


task3_dev_preds = hybrid_predict(dev_logits, temperature=best_T, use_prior=True)
task3_test_preds = hybrid_predict(test_logits, temperature=best_T, use_prior=True)

print("Task 3 hybrid dev accuracy:",
      round(seq_acc(task3_dev_preds, dev), 4))
print("Task 3 hybrid test accuracy:",
      round(seq_acc(task3_test_preds, test), 4))

print("\nAblation – test accuracy")
print("Hybrid, tuned temperature + prior:",
      round(seq_acc(task3_test_preds, test), 4))
print("Hybrid, T=1 + prior:",
      round(seq_acc(hybrid_predict(test_logits, 1.0, True), test), 4))
print("Hybrid, tuned temperature, no prior:",
      round(seq_acc(hybrid_predict(test_logits, best_T, False), test), 4))
print("Neural softmax argmax only:",
      round(argmax_accuracy(test_logits, test), 4))


Task 3 hybrid dev accuracy: 0.9368
Task 3 hybrid test accuracy: 0.9373

Ablation – test accuracy
Hybrid, tuned temperature + prior: 0.9373
Hybrid, T=1 + prior: 0.947
Hybrid, tuned temperature, no prior: 0.9437
Neural softmax argmax only: 0.9559


In [63]:
# Cell 12 – Final comparison, known/unknown word accuracy
train_vocab = set(word_counts)

def known_unknown_breakdown(predictions, data):
    known_correct = known_total = 0
    unknown_correct = unknown_total = 0

    for pred, (words_in_sentence, gold) in zip(predictions, data):
        for word, predicted, actual in zip(words_in_sentence, pred, gold):
            if word.lower() in train_vocab:
                known_total += 1
                known_correct += int(predicted == actual)
            else:
                unknown_total += 1
                unknown_correct += int(predicted == actual)

    overall = (known_correct + unknown_correct) / max(1, known_total + unknown_total)
    known_acc = known_correct / known_total if known_total else float("nan")
    unknown_acc = unknown_correct / unknown_total if unknown_total else float("nan")
    return overall, known_acc, unknown_acc


comparison = [
    ("Task 1: HMM counts", hmm_preds),
    ("Task 3: Neural-emission HMM", task3_test_preds),
]

if HAS_TASK2_CKPT:
    task2_model = BiLSTMCRF(BiLSTMEncoder(emb, len(chars)), K).to(device)
    task2_state = torch.load(CKPT2, map_location=device)
    task2_model.load_state_dict(task2_state, strict=True)
    task2_preds = predict_crf(task2_model, test)
    comparison.insert(1, ("Task 2: BiLSTM-CRF", task2_preds))
else:
    task2_preds = None
    print("Task 2 comparison omitted: original Task 2 checkpoint not found.")

print(f"{'Model':32s} {'Overall':>9s} {'Known':>9s} {'Unknown':>9s}")
results = {}
for model_name, predictions in comparison:
    overall, known_acc, unknown_acc = known_unknown_breakdown(predictions, test)
    results[model_name] = {
        "overall": overall,
        "known": known_acc,
        "unknown": unknown_acc
    }
    print(f"{model_name:32s} {overall:9.4f} {known_acc:9.4f} {unknown_acc:9.4f}")


Model                              Overall     Known   Unknown
Task 1: HMM counts                  0.9192    0.9321    0.7598
Task 2: BiLSTM-CRF                  0.9572    0.9650    0.8613
Task 3: Neural-emission HMM         0.9373    0.9482    0.8029


In [64]:
# Cell 13 – Confusion pairs and qualitative examples
task3_confusions = Counter()
task3_errors = []

for pred, (words_in_sentence, gold) in zip(task3_test_preds, test):
    for word, predicted, actual in zip(words_in_sentence, pred, gold):
        if predicted != actual:
            task3_confusions[(actual, predicted)] += 1
            task3_errors.append((word, actual, predicted))

print("Top 10 Task 3 confusions (gold -> predicted):")
for (gold_tag, predicted_tag), count in task3_confusions.most_common(10):
    print(f"{gold_tag:8s} -> {predicted_tag:8s} {count}")


def show_examples(title, condition, limit=3):
    print("\n===", title, "===")
    shown = 0
    for idx, (words_in_sentence, gold) in enumerate(test):
        h_pred = hmm_preds[idx]
        t3_pred = task3_test_preds[idx]
        t2_pred = task2_preds[idx] if task2_preds is not None else None

        if condition(h_pred, t2_pred, t3_pred, gold):
            for pos, (word, actual, h, t3) in enumerate(
                zip(words_in_sentence, gold, h_pred, t3_pred)
            ):
                t2_text = t2_pred[pos] if t2_pred is not None else "N/A"
                marker = " <--" if not (actual == h == t3 and
                                        (t2_pred is None or actual == t2_text)) else ""
                print(f"{word:16s} gold={actual:7s} HMM={h:7s} "
                      f"T2={t2_text:7s} T3={t3:7s}{marker}")
            print()
            shown += 1
            if shown >= limit:
                break
    if shown == 0:
        print("No example matched this condition in the test set.")


if task2_preds is not None:
    show_examples(
        "HMM wrong, Task 3 right",
        lambda h, t2, t3, gold: h != gold and t3 == gold
    )
    show_examples(
        "Task 3 wrong, Task 2 right",
        lambda h, t2, t3, gold: t3 != gold and t2 == gold
    )
else:
    show_examples(
        "HMM wrong, Task 3 right",
        lambda h, t2, t3, gold: h != gold and t3 == gold
    )

# Save reproducibility details and measured results.
output = {
    "seed": SEED,
    "tag_names": tag_names,
    "best_temperature": float(best_T),
    "task1_test": known_unknown_breakdown(hmm_preds, test),
    "task3_test": known_unknown_breakdown(task3_test_preds, test),
    "task3_dev_accuracy": float(seq_acc(task3_dev_preds, dev)),
    "task3_test_accuracy": float(seq_acc(task3_test_preds, test)),
    "task2_checkpoint_found": bool(HAS_TASK2_CKPT),
    "comparison": results,
    "ablation": {
        "hybrid_tuned_with_prior": float(seq_acc(task3_test_preds, test)),
        "hybrid_T1_with_prior": float(seq_acc(hybrid_predict(test_logits, 1.0, True), test)),
        "hybrid_tuned_without_prior": float(seq_acc(hybrid_predict(test_logits, best_T, False), test)),
        "neural_softmax_argmax": float(argmax_accuracy(test_logits, test))
    }
}

results_path = os.path.join(MYDRIVE, "nlp_assignment", "task3_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print("\nSaved results to:", results_path)
print(json.dumps(output, indent=2))


Top 10 Task 3 confusions (gold -> predicted):
NOUN     -> PROPN    159
PROPN    -> NOUN     144
PROPN    -> X        121
NOUN     -> X        87
ADP      -> ADV      71
NOUN     -> ADJ      71
VERB     -> NOUN     50
ADJ      -> NOUN     44
NOUN     -> VERB     41
ADJ      -> VERB     39

=== HMM wrong, Task 3 right ===
What             gold=PRON    HMM=PRON    T2=PRON    T3=PRON   
if               gold=SCONJ   HMM=SCONJ   T2=SCONJ   T3=SCONJ  
Google           gold=PROPN   HMM=PROPN   T2=PROPN   T3=PROPN  
Morphed          gold=VERB    HMM=PROPN   T2=VERB    T3=VERB    <--
Into             gold=ADP     HMM=ADP     T2=ADP     T3=ADP    
GoogleOS         gold=PROPN   HMM=PROPN   T2=PROPN   T3=PROPN  
?                gold=PUNCT   HMM=PUNCT   T2=PUNCT   T3=PUNCT  

[                gold=PUNCT   HMM=PUNCT   T2=PUNCT   T3=PUNCT  
via              gold=ADP     HMM=ADP     T2=ADP     T3=ADP    
Microsoft        gold=PROPN   HMM=PROPN   T2=PROPN   T3=PROPN  
Watch            gold=PROPN   HMM